# Feature Engineering Overview

**Purpose**: Test and visualize all 15 engineered features for quantitative trading.

**Feature Categories**:
1. Price/Momentum (5 features)
2. Volatility (3 features)
3. Mean Reversion (2 features)
4. Volume (2 features)
5. Cross-Asset (3 features)

---

In [1]:
# Import required libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from app.ml.features import compute_features, get_feature_list

# Set plotting style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 1. Load Stock Data (AAPL as Test Case)

In [2]:
# Load AAPL data
aapl_file = project_root / "data" / "raw" / "stocks" / "AAPL.csv"
spy_file = project_root / "data" / "raw" / "stocks" / "SPY.csv"

if not aapl_file.exists():
    raise FileNotFoundError(f"AAPL data not found. Run scripts/collect_data.py first.")

df_aapl = pd.read_csv(aapl_file)
print(f"✓ Loaded AAPL: {len(df_aapl)} rows")
print(f"  Date range: {df_aapl['Date'].min()} to {df_aapl['Date'].max()}")

# Load SPY data if available
spy_data = None
if spy_file.exists():
    spy_data = pd.read_csv(spy_file)
    print(f"✓ Loaded SPY: {len(spy_data)} rows")
else:
    print("⚠️  SPY data not found - cross-asset features will be skipped")

df_aapl.head()

✓ Loaded AAPL: 753 rows
  Date range: 2023-01-17 00:00:00-05:00 to 2026-01-15 00:00:00-05:00
✓ Loaded SPY: 754 rows


,Date,Open,High,Low,Close,Volume
0,2023-01-17 00:00:00-05:00,132.826128,135.249559,132.136535,133.919632,63646600
1,2023-01-18 00:00:00-05:00,134.786573,136.549963,133.023168,133.200500,69672800
2,2023-01-19 00:00:00-05:00,132.087296,134.225044,131.781906,133.259613,58280400
3,2023-01-20 00:00:00-05:00,133.269495,135.968779,132.225251,135.820999,80223600
4,2023-01-23 00:00:00-05:00,136.067249,141.189979,135.850518,139.012817,81760300


## 2. Compute All Features

In [3]:
# Compute features
df_features = compute_features(df_aapl, spy_data=spy_data)

print(f"✓ Computed features")
print(f"  Total columns: {len(df_features.columns)}")
print(f"  Total rows: {len(df_features)}")

# Display feature list
feature_list = get_feature_list()
present_features = [f for f in feature_list if f in df_features.columns]
print(f"\n✓ Features present: {len(present_features)}/{len(feature_list)}")

df_features.tail()

✓ Computed features
  Total columns: 21
  Total rows: 753

✓ Features present: 15/15


/Users/alexlgv/QuantSnap/app/ml/features.py:256: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_merged['spy_return_1d'] = df_merged['Close_spy'].pct_change(1) * 100
/Users/alexlgv/QuantSnap/app/ml/features.py:256: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_merged['spy_return_1d'] = df_merged['Close_spy'].pct_change(1) * 100
/Users/alexlgv/QuantSnap/app/ml/features.py:257: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or speci

,Date,Open,High,Low,Close,Volume,return_1d,return_5d,return_20d,price_sma50_ratio,...,atr_normalized,volatility_20d,rsi_14,price_deviation_from_sma20,bollinger_position,volume_ratio,volume_trend,correlation_spy_20d,relative_performance_spy_5d,spy_trend
748,2026-01-09 05:00:00+00:00,259.079987,260.209991,256.220001,259.369995,39997000,0.127388,-4.295050,-6.962481,-0.048460,...,1.504192,11.035948,21.935218,-4.138347,-0.964927,0.929516,-1323320.0,NaN,NaN,0
749,2026-01-12 05:00:00+00:00,259.160004,261.299988,256.799988,260.250000,45263800,0.339286,-2.622918,-6.394993,-0.044587,...,1.493616,11.308775,19.829135,-3.496020,-0.791158,1.037430,-2248940.0,NaN,NaN,0
750,2026-01-13 05:00:00+00:00,258.720001,261.809998,258.390015,261.049988,45730800,0.307392,-0.499313,-6.191610,-0.040940,...,1.490407,11.440808,25.614515,-2.889149,-0.658801,1.040742,-1031350.0,NaN,NaN,0
751,2026-01-14 05:00:00+00:00,259.489990,261.820007,256.709991,259.959991,40019400,-0.417543,-0.142125,-5.162159,-0.044232,...,1.556281,10.570004,19.061905,-3.039437,-0.671016,0.921658,-1506600.0,NaN,NaN,0
752,2026-01-15 05:00:00+00:00,260.649994,261.040009,257.049988,258.209991,39358400,-0.673181,-0.320420,-5.972104,-0.049927,...,1.587853,10.528543,11.651895,-3.396700,-0.726008,0.904654,-652160.0,NaN,NaN,0


## 3. Feature Documentation

### Price/Momentum Features (5 features)

| Feature | Description | Expected Range | Why It Matters |
|---------|-------------|----------------|----------------|
| `return_1d` | 1-day percentage return | -10% to +10% (typical) | Short-term momentum; captures immediate price changes |
| `return_5d` | 5-day percentage return | -15% to +15% (typical) | Weekly momentum; identifies short-term trends |
| `return_20d` | 20-day percentage return | -25% to +25% (typical) | Monthly momentum; shows intermediate trend strength |
| `price_sma50_ratio` | (Price / 50-day MA) - 1 | -0.2 to +0.2 | Distance from intermediate trend; overbought/oversold |
| `price_sma200_ratio` | (Price / 200-day MA) - 1 | -0.3 to +0.3 | Distance from long-term trend; bull/bear market signal |

### Volatility Features (3 features)

| Feature | Description | Expected Range | Why It Matters |
|---------|-------------|----------------|----------------|
| `atr_normalized` | ATR / Close price × 100 | 1% to 5% | Normalized volatility; higher = riskier |
| `volatility_20d` | Annualized 20-day volatility | 10% to 80% | Price stability measure; risk assessment |
| `rsi_14` | Relative Strength Index | 0 to 100 | Momentum oscillator; <30 oversold, >70 overbought |

### Mean Reversion Features (2 features)

| Feature | Description | Expected Range | Why It Matters |
|---------|-------------|----------------|----------------|
| `price_deviation_from_sma20` | % deviation from 20-day MA | -15% to +15% | Mean reversion signal; extreme values suggest reversal |
| `bollinger_position` | Position within Bollinger Bands | -1 to +1 | Standardized overbought/oversold; -1=lower band, +1=upper band |

### Volume Features (2 features)

| Feature | Description | Expected Range | Why It Matters |
|---------|-------------|----------------|----------------|
| `volume_ratio` | Volume / 20-day avg volume | 0.2 to 3.0 | Volume strength; >1.5 = unusual activity |
| `volume_trend` | 5-day volume slope | negative to positive | Volume momentum; confirms price moves |

### Cross-Asset Features (3 features)

| Feature | Description | Expected Range | Why It Matters |
|---------|-------------|----------------|----------------|
| `correlation_spy_20d` | 20-day rolling correlation with SPY | -1 to +1 | Market relationship; high correlation = follows market |
| `relative_performance_spy_5d` | Stock return - SPY return (5d) | -20% to +20% | Outperformance vs market; positive = beating S&P 500 |
| `spy_trend` | Binary: SPY > 200-day SMA? | 0 or 1 | Market regime; 1 = bull market, 0 = bear market |

## 4. Feature Statistics

In [4]:
# Display statistics for all features
feature_cols = [col for col in get_feature_list() if col in df_features.columns]

print("Feature Statistics (excluding NaN values):")
print("=" * 80)

stats_df = df_features[feature_cols].describe().round(3)
stats_df

Feature Statistics (excluding NaN values):


,return_1d,return_5d,return_20d,price_sma50_ratio,price_sma200_ratio,atr_normalized,volatility_20d,rsi_14,price_deviation_from_sma20,bollinger_position,volume_ratio,volume_trend,correlation_spy_20d,relative_performance_spy_5d,spy_trend
count,752.000,748.000,733.000,753.000,753.000,753.000,751.000,752.000,753.000,753.000,753.000,7.520000e+02,0.0,0.0,753.0
mean,0.100,0.509,1.915,0.022,0.069,2.091,23.240,54.755,0.862,0.164,1.000,-2.304992e+04,NaN,NaN,0.0
std,1.607,3.734,6.347,0.057,0.094,0.757,11.001,18.109,3.643,0.638,0.395,8.038017e+06,NaN,NaN,0.0
min,-9.246,-22.747,-21.925,-0.240,-0.245,1.184,6.526,0.000,-18.610,-1.598,0.405,-5.759237e+07,NaN,NaN,0.0
25%,-0.670,-1.652,-2.292,-0.018,0.013,1.672,16.876,41.352,-1.489,-0.356,0.776,-3.194235e+06,NaN,NaN,0.0
50%,0.119,0.573,2.166,0.026,0.074,1.933,21.974,56.814,1.247,0.285,0.916,-2.793500e+05,NaN,NaN,0.0
75%,0.840,2.662,6.402,0.065,0.147,2.262,26.004,68.031,3.354,0.681,1.112,3.139992e+06,NaN,NaN,0.0
max,15.329,17.237,20.640,0.176,0.261,6.985,81.810,96.163,10.018,1.701,5.246,5.399070e+07,NaN,NaN,0.0


In [5]:
# Check for missing values
print("\nMissing Values per Feature:")
print("=" * 80)

missing_df = pd.DataFrame({
    'Feature': feature_cols,
    'Missing': [df_features[col].isna().sum() for col in feature_cols],
    'Missing %': [(df_features[col].isna().sum() / len(df_features)) * 100 for col in feature_cols]
})

missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

if len(missing_df) > 0:
    display(missing_df)
else:
    print("✓ No missing values in any features!")


Missing Values per Feature:


,Feature,Missing,Missing %
12,correlation_spy_20d,753,100.000000
13,relative_performance_spy_5d,753,100.000000
2,return_20d,20,2.656042
1,return_5d,5,0.664011
6,volatility_20d,2,0.265604
0,return_1d,1,0.132802
7,rsi_14,1,0.132802
11,volume_trend,1,0.132802


## 5. Mean Reversion Features Visualization

Visualize price vs SMA20 and highlight oversold/overbought zones using Bollinger Bands.

In [ ]:
# Prepare data for visualization (last 120 days)
df_viz = df_features.tail(120).copy()
df_viz['Date'] = pd.to_datetime(df_viz['Date'])

# Calculate SMA20 and Bollinger Bands
df_viz['sma_20'] = df_viz['Close'].rolling(window=20).mean()
df_viz['std_20'] = df_viz['Close'].rolling(window=20).std()
df_viz['upper_band'] = df_viz['sma_20'] + (2 * df_viz['std_20'])
df_viz['lower_band'] = df_viz['sma_20'] - (2 * df_viz['std_20'])

# Create the plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot 1: Price with Bollinger Bands
ax1 = axes[0]
ax1.plot(df_viz['Date'], df_viz['Close'], label='Close Price', color='black', linewidth=2)
ax1.plot(df_viz['Date'], df_viz['sma_20'], label='SMA(20)', color='blue', linewidth=1.5, linestyle='--')
ax1.plot(df_viz['Date'], df_viz['upper_band'], label='Upper Band (+2σ)', color='red', linewidth=1, linestyle=':')
ax1.plot(df_viz['Date'], df_viz['lower_band'], label='Lower Band (-2σ)', color='green', linewidth=1, linestyle=':')

# Fill between bands
ax1.fill_between(df_viz['Date'], df_viz['upper_band'], df_viz['lower_band'], alpha=0.1, color='gray')

# Highlight overbought zones (price near upper band)
overbought = df_viz[df_viz['bollinger_position'] > 0.8]
ax1.scatter(overbought['Date'], overbought['Close'], color='red', s=50, alpha=0.6, 
           label='Overbought (BB > 0.8)', zorder=5)

# Highlight oversold zones (price near lower band)
oversold = df_viz[df_viz['bollinger_position'] < -0.8]
ax1.scatter(oversold['Date'], oversold['Close'], color='green', s=50, alpha=0.6, 
           label='Oversold (BB < -0.8)', zorder=5)

ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title('AAPL: Mean Reversion - Price vs Bollinger Bands (Last 120 Days)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Mean Reversion Indicators
ax2 = axes[1]

# Bollinger Position
ax2.plot(df_viz['Date'], df_viz['bollinger_position'], label='Bollinger Position', 
        color='purple', linewidth=2)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax2.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Upper Band')
ax2.axhline(y=-1, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Lower Band')
ax2.axhline(y=0.8, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax2.axhline(y=-0.8, color='green', linestyle=':', linewidth=1, alpha=0.5)

# Fill overbought/oversold zones
ax2.fill_between(df_viz['Date'], 0.8, 1.2, alpha=0.2, color='red', label='Overbought Zone')
ax2.fill_between(df_viz['Date'], -0.8, -1.2, alpha=0.2, color='green', label='Oversold Zone')

ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Bollinger Position', fontsize=12)
ax2.set_title('Mean Reversion Indicator', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  • Overbought (red): Price near upper Bollinger Band → potential reversal down")
print("  • Oversold (green): Price near lower Bollinger Band → potential reversal up")
print("  • Bollinger Position: -1 to +1 normalized position within bands")

## 6. Price Deviation from SMA20

In [ ]:
# Plot price deviation from SMA20
fig, ax = plt.subplots(figsize=(14, 6))

# Plot deviation
ax.plot(df_viz['Date'], df_viz['price_deviation_from_sma20'], 
       color='darkblue', linewidth=2, label='Price Deviation from SMA20')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)

# Highlight extreme deviations
ax.axhline(y=5, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='±5% threshold')
ax.axhline(y=-5, color='orange', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1, alpha=0.7, label='±10% threshold')
ax.axhline(y=-10, color='red', linestyle='--', linewidth=1, alpha=0.7)

# Fill zones
ax.fill_between(df_viz['Date'], 5, 10, alpha=0.1, color='orange')
ax.fill_between(df_viz['Date'], -5, -10, alpha=0.1, color='orange')
ax.fill_between(df_viz['Date'], 10, 20, alpha=0.1, color='red')
ax.fill_between(df_viz['Date'], -10, -20, alpha=0.1, color='red')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Deviation (%)', fontsize=12)
ax.set_title('AAPL: Price Deviation from 20-Day Moving Average', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  • Positive deviation: Price above SMA20 (uptrend)")
print("  • Negative deviation: Price below SMA20 (downtrend)")
print("  • Extreme deviations (>±10%): Strong mean reversion candidates")

## 7. Volume Features Visualization

In [ ]:
# Plot volume features
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Plot 1: Volume Ratio
ax1 = axes[0]
ax1.bar(df_viz['Date'], df_viz['volume_ratio'], color='steelblue', alpha=0.7, label='Volume Ratio')
ax1.axhline(y=1.0, color='black', linestyle='-', linewidth=1.5, alpha=0.7, label='Average Volume')
ax1.axhline(y=1.5, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='High Volume (1.5x)')
ax1.axhline(y=2.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Very High Volume (2x)')

ax1.set_ylabel('Volume Ratio', fontsize=12)
ax1.set_title('AAPL: Volume Ratio (Volume / 20-day Average)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Volume Trend
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in df_viz['volume_trend']]
ax2.bar(df_viz['Date'], df_viz['volume_trend'], color=colors, alpha=0.7, label='Volume Trend (5d slope)')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)

ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Volume Trend', fontsize=12)
ax2.set_title('Volume Trend (5-Day Slope)', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  • Volume Ratio > 1.5: Unusual volume activity (potential breakout/breakdown)")
print("  • Volume Trend > 0: Increasing volume (green) - confirms price moves")
print("  • Volume Trend < 0: Decreasing volume (red) - weakening momentum")

## 8. Cross-Asset Features (if SPY data available)

In [ ]:
if spy_data is not None and 'correlation_spy_20d' in df_features.columns:
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    # Plot 1: Correlation with SPY
    ax1 = axes[0]
    ax1.plot(df_viz['Date'], df_viz['correlation_spy_20d'], color='darkgreen', linewidth=2)
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.5)
    ax1.axhline(y=0.7, color='green', linestyle='--', linewidth=1, alpha=0.7, label='High correlation (0.7)')
    ax1.fill_between(df_viz['Date'], 0.7, 1.0, alpha=0.1, color='green')
    ax1.set_ylabel('Correlation', fontsize=12)
    ax1.set_title('AAPL vs SPY: 20-Day Rolling Correlation', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(-1, 1)
    
    # Plot 2: Relative Performance
    ax2 = axes[1]
    colors = ['green' if x > 0 else 'red' for x in df_viz['relative_performance_spy_5d']]
    ax2.bar(df_viz['Date'], df_viz['relative_performance_spy_5d'], color=colors, alpha=0.7)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.7)
    ax2.set_ylabel('Relative Return (%)', fontsize=12)
    ax2.set_title('Relative Performance vs SPY (5-Day)', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Market Regime (SPY Trend)
    ax3 = axes[2]
    ax3.fill_between(df_viz['Date'], 0, df_viz['spy_trend'], 
                     color='green', alpha=0.3, label='Bull Market (SPY > SMA200)')
    ax3.fill_between(df_viz['Date'], df_viz['spy_trend'], 1, 
                     color='red', alpha=0.3, label='Bear Market (SPY < SMA200)')
    ax3.set_xlabel('Date', fontsize=12)
    ax3.set_ylabel('Market Regime', fontsize=12)
    ax3.set_title('SPY Market Regime (1 = Bull, 0 = Bear)', fontsize=12, fontweight='bold')
    ax3.set_ylim(-0.1, 1.1)
    ax3.set_yticks([0, 1])
    ax3.set_yticklabels(['Bear', 'Bull'])
    ax3.legend(loc='upper left', fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Interpretation:")
    print("  • High correlation (>0.7): Stock follows market closely")
    print("  • Positive relative performance: Stock outperforming S&P 500")
    print("  • Bull market regime: SPY above 200-day SMA (favorable conditions)")
else:
    print("⚠️  Cross-asset features not available. Download SPY data to enable.")

## 9. Feature Correlation Matrix

In [ ]:
# Compute correlation matrix
feature_cols = [col for col in get_feature_list() if col in df_features.columns]
corr_matrix = df_features[feature_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
           square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("  • High correlation (>0.8): Features may be redundant")
print("  • Negative correlation (<-0.5): Features provide opposite signals")
print("  • Low correlation (<0.3): Features provide independent information")

## 10. Summary

### ✅ Feature Verification Checklist

- [ ] All 15 features computed successfully
- [ ] Feature ranges are within expected bounds
- [ ] Missing values are minimal and expected (rolling windows)
- [ ] Mean reversion features correctly identify overbought/oversold zones
- [ ] Volume features capture unusual activity
- [ ] Cross-asset features show market relationship (if SPY available)

### 📈 Next Steps

1. Run this notebook on other stocks to verify consistency
2. Analyze feature distributions across the entire universe
3. Investigate feature importance for predictive modeling
4. Test features on different market conditions (bull/bear/sideways)

---

**Note**: This notebook validates that `compute_features()` works correctly and all features make intuitive sense.